# 🛰️ Sentinel-5P & Sentinel-2 Multimodal Satellite Data Acquisition Pipeline
### Project: *Satellite-Driven Multi-Pollutant Forecasting for Industrial & Respiratory Disease Risk Mapping*
**Pipeline Module:** Standardized Atmospheric & Optical Extraction (2019–2024)  
**Geospatial Scope:** Chennai Industrial Corridor `[79.95°E, 12.80°N, 80.45°E, 13.30°N]` (~55 km × 55 km)  
**Target Platform:** Google Colab / Google Earth Engine (Python API)  
**Google Drive Destination:** [Drive Folder (ID: 1N9E8EzMA2ku7Y7lasMPu7oSupz5uxZHh)](https://drive.google.com/drive/folders/1N9E8EzMA2ku7Y7lasMPu7oSupz5uxZHh?usp=drive_link)

---

### ⚙️ Pipeline Specifications & Quality Benchmarks

* **Region of Interest (AOI):** Chennai Industrial Corridor `[79.95°E, 12.80°N, 80.45°E, 13.30°N]` (~55 km × 55 km). Encompasses Manali Petrochemical cluster, Ennore Thermal Power Station, Ambattur/Guindy industrial zones, and Chennai Port.
* **Temporal Window:** 5-day median/mean composites from **2019-01-01 to 2024-12-31** (~438 uniform windows).
* **S2 Channels (12):** B2, B3, B4, B5, B6, B7, B8, B11, B12 + NDVI, NDBI, NDMI.
* **S5P Channels (3):** NO2, CO, SO2 (all QA-filtered > 0.5) + per-gas overpass counts.
* **Strict Nodata Convention:** Missing pixels written as `-9999.0` (zeros are never conflated with clean air).
* **Physical Outlier Handling:** Small negative retrieval noise around zero preserved; unphysical anomalies masked.
* **Storage Footprint:** ~4.5 GB total for S2 + ~40 MB for S5P (comfortably fits on Google Drive).

---

### 🚀 Execution Options
* **Option A (Cell 8A):** Direct interactive Colab download with live progress bar.
* **Option B (Cell 8B):** **Fire-and-Forget Cloud Cluster Mode (`ee.batch.Export.image.toDrive`)**. Runs 100% on Google's cloud servers. You can submit the tasks, close Colab, disconnect, and shut down your laptop; Google Cloud will populate your Drive automatically!
* **Option C (Cell 8C):** Cloud Cluster Task Monitor to verify completed exports anytime.


In [ ]:

# Cell 1: Dependencies, Drive (Colab), and Earth Engine
import sys
import subprocess

for pkg, mod in [
    ("earthengine-api", "ee"),
    ("rasterio", "rasterio"),
    ("tqdm", "tqdm"),
    ("pandas", "pandas"),
]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg]
        )

import os
import io
import time
import json
import datetime
import math
import threading
import concurrent.futures

import numpy as np
import pandas as pd
import rasterio
from rasterio.io import MemoryFile
import requests
from tqdm.auto import tqdm
import ee


# -------------------------------------------------------------------------
# Earth Engine Project & Google Drive Destination Configuration
# -------------------------------------------------------------------------
EE_PROJECT = "final-year-projects-501407"

# Target Google Drive Folder ID
TARGET_DRIVE_FOLDER_ID = "1N9E8EzMA2ku7Y7lasMPu7oSupz5uxZHh"

# Target folder name in Google Drive
TARGET_FOLDER_NAME = "Chennai_Satellite_Dataset"


# -------------------------------------------------------------------------
# Configure Google Drive or Local Storage
# -------------------------------------------------------------------------
try:
    from google.colab import drive, userdata

    drive.mount("/content/drive", force_remount=False)

    mydrive = "/content/drive/MyDrive"
    target_path = os.path.join(mydrive, TARGET_FOLDER_NAME)

    os.makedirs(target_path, exist_ok=True)

    BASE_DIR = target_path

    print(f"✓ Connected to Google Drive destination: {BASE_DIR}")
    print(f"✓ Target Drive Folder ID: {TARGET_DRIVE_FOLDER_ID}")

except ImportError:
    BASE_DIR = os.path.abspath("./Chennai_Satellite_Dataset")
    os.makedirs(BASE_DIR, exist_ok=True)

    print(f"✓ Running in local environment. Base directory: {BASE_DIR}")


# -------------------------------------------------------------------------
# Robust Earth Engine Authentication & Initialization
# -------------------------------------------------------------------------
def init_earth_engine(project_id):
    # Step 1: Try initializing directly if a valid token exists
    try:
        ee.Initialize(project=project_id)
        print(
            f"✓ Earth Engine initialized successfully "
            f"with project: '{project_id}'"
        )
        return True

    except Exception as e:
        print(f"ℹ️ Earth Engine authentication required: {e}")

    # Step 2: Authenticate
    print("\n" + "=" * 70)
    print("EARTH ENGINE AUTHORIZATION:")
    print("Open the generated link, grant permissions,")
    print("copy the verification code, and paste it when prompted.")
    print("=" * 70 + "\n")

    try:
        ee.Authenticate(auth_mode="notebook")

    except Exception as auth_err:
        print(
            f"Notice: notebook authentication raised ({auth_err}). "
            "Trying default authentication..."
        )

        try:
            ee.Authenticate()

        except Exception as colab_err:
            print(f"Default authorization failed: {colab_err}")
            print("Trying a fresh notebook authorization...")

            ee.Authenticate(auth_mode="notebook", force=True)

    # Step 3: Initialize with the configured project ID
    try:
        ee.Initialize(project=project_id)
        print(
            f"\n✓ Earth Engine initialized successfully "
            f"with project: '{project_id}'"
        )
        return True

    except Exception as init_err:
        print(
            f"Could not initialize with project '{project_id}': "
            f"{init_err}"
        )

        # Step 4: Try initialization with default credentials
        try:
            ee.Initialize()
            print(
                "✓ Earth Engine initialized with default "
                "project credentials."
            )
            return True

        except Exception:
            user_proj = input(
                "Enter your Earth Engine Google Cloud Project ID "
                "(e.g. final-year-projects-501407): "
            ).strip()

            if user_proj:
                ee.Initialize(project=user_proj)
                print(
                    "✓ Earth Engine initialized successfully "
                    f"with project: '{user_proj}'"
                )
                return True

            raise


# -------------------------------------------------------------------------
# Initialize Pipeline
# -------------------------------------------------------------------------
init_earth_engine(EE_PROJECT)

print("✓ Pipeline ready. Storage directory configured at:", BASE_DIR)


Mounted at /content/drive
✓ Connected to Google Drive destination: /content/drive/MyDrive/Chennai_Satellite_Dataset
✓ Target Drive Folder ID: 1N9E8EzMA2ku7Y7lasMPu7oSupz5uxZHh
ℹ️ Earth Engine authentication required: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

EARTH ENGINE AUTHORIZATION:
Open the generated link, grant permissions,
copy the verification code, and paste it when prompted.

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=Jf94YAO7XtK_cFqCrN_wuH

In [ ]:
# Cell 2: configuration (no Earth Engine calls here)
REGIONS = {
    # [min_lon, min_lat, max_lon, max_lat]
    "chennai": [79.95, 12.80, 80.45, 13.30],      # Chennai Industrial Corridor (~55 km x 55 km)
    "tamil_nadu": [76.20, 8.05, 80.40, 13.55],    # Full Tamil Nadu state incl. coast
}

# -------------------------------------------------------------------------
# Active Region: CHENNAI INDUSTRIAL CORRIDOR
# (Manali Petrochemical, Ennore Thermal, Ambattur/Guindy, Chennai Port)
# -------------------------------------------------------------------------
REGION = "chennai"

CONFIG = {
    "BBOX": REGIONS[REGION],
    "START_DATE": "2019-01-01",
    "END_DATE": "2024-12-31",
    "WINDOW_DAYS": 5,
    "S5P_CELL_DEG": 0.05,          # ~5.5 km, close to TROPOMI footprint (5.5 x 3.5 km)
    "S2_PER_S5P": 8,               # S2 cell = 0.05/8 = 0.00625 deg (~690 m), nests 8x8 in each S5P cell
    "S2_FILL_PAST_DAYS": 0,        # 0 = window only. >0 fills gaps from the PAST only (never future)
    "MAX_S2_CLOUD_PERCENT": 85,    # scene filter; per-pixel SCL masking does real filtering
    "NODATA": -9999.0,
    "WORKERS": 2,                  # 2 workers optimal for GEE rate-limit resilience
    "OUTPUT_DIR": os.path.join(BASE_DIR, f"data_v3_{REGION}"),
}

# Outlier floors (mol/m^2). Outliers below are masked; small retrieval noise around zero is kept.
GASES = {
    "NO2": ("COPERNICUS/S5P/OFFL/L3_NO2", "tropospheric_NO2_column_number_density", -0.0005),
    "CO":  ("COPERNICUS/S5P/OFFL/L3_CO",  "CO_column_number_density",              -0.0010),
    "SO2": ("COPERNICUS/S5P/OFFL/L3_SO2", "SO2_column_number_density",             -0.0050),
}
S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]
S2_CLEAR_EXCLUDE = [0, 1, 3, 8, 9, 10, 11]   # SCL: no data, saturated, shadow, cloud med/high, cirrus, snow

def make_grid(bbox, cell):
    # Snap the box outward to whole cells so every run, region and sensor shares one lattice.
    x0 = math.floor(bbox[0] / cell + 1e-9) * cell
    y1 = math.ceil(bbox[3] / cell - 1e-9) * cell
    w = int(round((math.ceil(bbox[2] / cell - 1e-9) * cell - x0) / cell))
    h = int(round((y1 - math.floor(bbox[1] / cell + 1e-9) * cell) / cell))
    return [cell, 0, round(x0, 6), 0, -cell, round(y1, 6)], w, h

S5P_TRANSFORM, S5P_W, S5P_H = make_grid(CONFIG["BBOX"], CONFIG["S5P_CELL_DEG"])
S2_CELL = CONFIG["S5P_CELL_DEG"] / CONFIG["S2_PER_S5P"]
S2_TRANSFORM = [S2_CELL, 0, S5P_TRANSFORM[2], 0, -S2_CELL, S5P_TRANSFORM[5]]
S2_W, S2_H = S5P_W * CONFIG["S2_PER_S5P"], S5P_H * CONFIG["S2_PER_S5P"]

S5P_DIR = os.path.join(CONFIG["OUTPUT_DIR"], "s5p_composites")
S2_DIR = os.path.join(CONFIG["OUTPUT_DIR"], "s2_composites")
MANIFEST = os.path.join(CONFIG["OUTPUT_DIR"], "dataset_manifest.csv")
os.makedirs(S5P_DIR, exist_ok=True)
os.makedirs(S2_DIR, exist_ok=True)

print("=" * 70)
print(f"ACTIVE REGION: {REGION.upper()} (CHENNAI INDUSTRIAL CORRIDOR)")
print(f"Bounding Box:  {CONFIG['BBOX']}")
print(f"S5P Grid:      {S5P_W}x{S5P_H} pixels at {CONFIG['S5P_CELL_DEG']} deg (~5.5 km)")
print(f"S2 Grid:       {S2_W}x{S2_H} pixels at {round(S2_CELL, 6)} deg (~690 m)")
print(f"Google Drive Destination: {CONFIG['OUTPUT_DIR']}")
print("=" * 70)


ACTIVE REGION: CHENNAI (CHENNAI INDUSTRIAL CORRIDOR)
Bounding Box:  [79.95, 12.8, 80.45, 13.3]
S5P Grid:      10x10 pixels at 0.05 deg (~5.5 km)
S2 Grid:       80x80 pixels at 0.00625 deg (~690 m)
Google Drive Destination: /content/drive/MyDrive/Chennai_Satellite_Dataset/data_v3_chennai


In [ ]:
# Cell 3: 5-day windows (same anchor as the old export: 2019-01-01, 438 windows)
def generate_time_windows(start, end, days):
    s = datetime.datetime.strptime(start, "%Y-%m-%d")
    e = datetime.datetime.strptime(end, "%Y-%m-%d")
    out, cur = [], s
    while cur + datetime.timedelta(days=days) <= e:
        out.append({"window_id": cur.strftime("%Y%m%d"), "start_date": cur.strftime("%Y-%m-%d"),
                    "end_date": (cur + datetime.timedelta(days=days)).strftime("%Y-%m-%d")})
        cur += datetime.timedelta(days=days)
    return out

TIME_WINDOWS = generate_time_windows(CONFIG["START_DATE"], CONFIG["END_DATE"], CONFIG["WINDOW_DAYS"])
print(len(TIME_WINDOWS), "windows:", TIME_WINDOWS[0]["start_date"], "...", TIME_WINDOWS[-1]["start_date"])

438 windows: 2019-01-01 ... 2024-12-25


In [ ]:
# Cell 4: Sentinel-5P composite: masked mean + overpass count, area-averaged onto the 0.05 deg grid
CRS = "EPSG:4326"
AOI = ee.Geometry.Rectangle(CONFIG["BBOX"])
NODATA = CONFIG["NODATA"]

def s5p_gas(gas, d0, d1):
    coll, band, floor = GASES[gas]
    col = ee.ImageCollection(coll).filterBounds(AOI).filterDate(d0, d1).select([band], ["v"])
    n_images = col.size().getInfo()
    if n_images == 0:
        empty = ee.Image.constant([NODATA, 0]).rename([gas, gas + "_n"]).toFloat()
        return empty.updateMask(ee.Image.constant([0, 1])), 0
    native = col.first().projection()            # a composite has no projection of its own; set it explicitly
    col = col.map(lambda im: im.updateMask(im.gte(floor)))
    mean = col.mean().setDefaultProjection(native)
    count = col.count().unmask(0).setDefaultProjection(native)
    to_grid = lambda im: (im.reduceResolution(ee.Reducer.mean(), maxPixels=1024)
                            .reproject(crs=CRS, crsTransform=S5P_TRANSFORM))
    return to_grid(mean).rename(gas).addBands(to_grid(count).rename(gas + "_n")).toFloat(), n_images

def get_s5p_composite(start_date, end_date):
    # Band order: NO2, CO, SO2, NO2_n, CO_n, SO2_n. Missing values are -9999 (never 0).
    parts = {g: s5p_gas(g, start_date, end_date) for g in GASES}
    if all(n == 0 for _, n in parts.values()):
        return None, {g: 0 for g in GASES}
    img = ee.Image.cat([parts[g][0].select(g) for g in GASES] + [parts[g][0].select(g + "_n") for g in GASES])
    return img.unmask(NODATA).toFloat(), {g: n for g, (_, n) in parts.items()}

In [ ]:
# Cell 5: Sentinel-2 composite: window only (or past-only fill), clouds masked, gaps = -9999
def mask_s2(img):
    scl = img.select("SCL")
    clear = ee.Image.constant(1)
    for c in S2_CLEAR_EXCLUDE:
        clear = clear.And(scl.neq(c))
    return img.select(S2_BANDS).divide(10000.0).updateMask(clear)

def get_s2_composite(start_date, end_date):
    d0 = ee.Date(start_date).advance(-CONFIG["S2_FILL_PAST_DAYS"], "day")
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(AOI).filterDate(d0, ee.Date(end_date))
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CONFIG["MAX_S2_CLOUD_PERCENT"])))
    if col.size().getInfo() == 0:
        return None
    comp = col.map(mask_s2).median()
    ndvi = comp.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndbi = comp.normalizedDifference(["B11", "B8"]).rename("NDBI")
    ndmi = comp.normalizedDifference(["B8", "B11"]).rename("NDMI")
    # NDMI = -NDBI exactly (same two bands); kept only so the 12-band layout matches the old files.
    return comp.addBands([ndvi, ndbi, ndmi]).unmask(NODATA).toFloat()

In [ ]:
# Cell 6: download straight onto the fixed grid, declare nodata, tag the file as v2
S5P_DESC = list(GASES) + [g + "_n" for g in GASES]
S2_DESC = S2_BANDS + ["NDVI", "NDBI", "NDMI"]

def is_v2_file(path):
    try:
        with rasterio.open(path) as src:
            return src.tags().get("pipeline") == "v2" and src.nodata == NODATA
    except Exception:
        return False

def download(img, path, transform, w, h, descriptions, max_retries=6):
    for attempt in range(1, max_retries + 1):
        try:
            url = img.getDownloadURL({"format": "GEO_TIFF", "crs": CRS, "crs_transform": transform,
                                      "dimensions": f"{w}x{h}"})
            r = requests.get(url, timeout=300)
            if r.status_code == 429 or r.status_code >= 500:
                raise RuntimeError(f"HTTP {r.status_code}")
            r.raise_for_status()
            with MemoryFile(r.content) as mem, mem.open() as src:
                a, prof = src.read().astype("float32"), src.profile
            assert (prof["width"], prof["height"]) == (w, h), f"unexpected size {prof['width']}x{prof['height']}"
            assert a.shape[0] == len(descriptions), f"expected {len(descriptions)} bands, got {a.shape[0]}"
            a[~np.isfinite(a)] = NODATA
            prof.update(dtype="float32", nodata=NODATA, compress="deflate", driver="GTiff")
            tmp = path + ".tmp"
            with rasterio.open(tmp, "w", **prof) as dst:
                dst.write(a)
                dst.descriptions = tuple(descriptions)
                dst.update_tags(pipeline="v2", created=datetime.datetime.utcnow().isoformat(), region=REGION)
            os.replace(tmp, path)
            return a
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(min(60, 2 ** attempt))

In [ ]:
# Cell 7: smoke test on one window. Must print SMOKE TEST PASSED before running cell 8.
w = {"window_id": "20200201", "start_date": "2020-02-01", "end_date": "2020-02-06"}
img, n_images = get_s5p_composite(w["start_date"], w["end_date"])
assert img is not None, "no S5P images in the smoke-test window"
p = os.path.join(S5P_DIR, "smoke_s5p.tif")
a = download(img, p, S5P_TRANSFORM, S5P_W, S5P_H, S5P_DESC)
vals, counts = a[:3], a[3:]
print("images per gas:", n_images)
for i, g in enumerate(GASES):
    v = vals[i]
    ok = v != NODATA
    print(f"{g}: valid {ok.mean():.0%}, exact zeros {np.mean(v[ok] == 0):.2%}, negatives {np.mean(v[ok] < 0):.1%}, "
          f"mean {v[ok].mean():.3e}, overpasses max {counts[i].max():.1f}")
    assert ok.any(), f"{g}: no valid pixels"
    assert np.all(v[counts[i] == 0] == NODATA), f"{g}: a value where no overpass was counted"
    assert np.mean(v[ok] == 0) < 0.01, f"{g}: exact zeros present, missing data is leaking in as 0"
co = vals[1][vals[1] != NODATA]
assert 0.01 < np.median(co) < 0.1, f"CO median {np.median(co):.3g} mol/m2 is not plausible"

s2 = get_s2_composite(w["start_date"], w["end_date"])
if s2 is not None:
    b = download(s2, os.path.join(S2_DIR, "smoke_s2.tif"), S2_TRANSFORM, S2_W, S2_H, S2_DESC)
    clear = b[0] != NODATA
    print(f"S2: clear {clear.mean():.0%}, B4 median {np.median(b[2][clear]):.3f}")
    assert np.all((b[:9][:, clear] > -0.1) & (b[:9][:, clear] < 2.0)), "S2 reflectance out of range"
else:
    print("S2: no scenes in this window (allowed)")
os.remove(p)
print("SMOKE TEST PASSED")

/tmp/ipykernel_524/2769490694.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  dst.update_tags(pipeline="v2", created=datetime.datetime.utcnow().isoformat(), region=REGION)


images per gas: {'NO2': 70, 'CO': 70, 'SO2': 70}
NO2: valid 100%, exact zeros 0.00%, negatives 0.0%, mean 3.495e-05, overpasses max 5.0
CO: valid 95%, exact zeros 0.00%, negatives 0.0%, mean 3.713e-02, overpasses max 5.0
SO2: valid 100%, exact zeros 0.00%, negatives 21.0%, mean 8.666e-05, overpasses max 5.0


S2: clear 95%, B4 median 0.082
SMOKE TEST PASSED


In [ ]:
# Cell 8A: Option A - Direct Colab Download (Interactive with live progress)
lock = threading.Lock()
records = {}
if os.path.exists(MANIFEST):
    for r in pd.read_csv(MANIFEST, dtype={"window_id": str}).to_dict("records"):
        records[r["window_id"]] = r

def valid_fraction(a, i):
    return float(np.mean(a[i] != NODATA)) if a is not None else 0.0

def process(win):
    wid = win["window_id"]
    s5p_f, s2_f = f"s5p_{wid}.tif", f"s2_{wid}.tif"
    s5p_p, s2_p = os.path.join(S5P_DIR, s5p_f), os.path.join(S2_DIR, s2_f)
    rec = {**win, "s5p_file": s5p_f, "s2_file": s2_f, "pipeline": "v2",
           "timestamp": datetime.datetime.utcnow().isoformat()}
    try:
        if is_v2_file(s5p_p):
            with rasterio.open(s5p_p) as src:
                a = src.read()
            rec["s5p_status"] = "ok"
        else:
            img, n = get_s5p_composite(win["start_date"], win["end_date"])
            a = download(img, s5p_p, S5P_TRANSFORM, S5P_W, S5P_H, S5P_DESC) if img is not None else None
            rec["s5p_status"] = "ok" if a is not None else "no_s5p_data"
        for i, g in enumerate(GASES):
            rec[f"valid_{g}"] = valid_fraction(a, i)
    except Exception as e:
        rec["s5p_status"] = f"err: {str(e)[:60]}"
    try:
        if is_v2_file(s2_p):
            rec["s2_status"] = "ok"
        else:
            s2 = get_s2_composite(win["start_date"], win["end_date"])
            if s2 is None:
                rec["s2_status"] = "cloud_gap"
            else:
                b = download(s2, s2_p, S2_TRANSFORM, S2_W, S2_H, S2_DESC)
                rec["s2_status"] = "ok"
                rec["s2_clear"] = float(np.mean(b[0] != NODATA))
    except Exception as e:
        rec["s2_status"] = f"err: {str(e)[:60]}"
    ok5, ok2 = rec["s5p_status"] == "ok", rec["s2_status"] == "ok"
    rec["status"] = "complete" if ok5 and ok2 else ("s5p_only" if ok5 else "failed")
    with lock:
        records[wid] = rec
    return rec

todo = [w for w in TIME_WINDOWS if records.get(w["window_id"], {}).get("status") not in ("complete", "s5p_only")
        or not is_v2_file(os.path.join(S5P_DIR, f"s5p_{w['window_id']}.tif"))]
print(f"{len(TIME_WINDOWS) - len(todo)} windows already done, {len(todo)} to process")

def save_manifest():
    with lock:
        df = pd.DataFrame(sorted(records.values(), key=lambda r: r["start_date"]))
    df.to_csv(MANIFEST, index=False)
    return df

with concurrent.futures.ThreadPoolExecutor(max_workers=CONFIG["WORKERS"]) as ex:
    futures = [ex.submit(process, w) for w in todo]
    for k, f in enumerate(tqdm(concurrent.futures.as_completed(futures), total=len(futures))):
        f.result()
        if (k + 1) % 10 == 0:
            save_manifest()
df = save_manifest()
print(df.status.value_counts())
failed = df[df.status == "failed"]
if len(failed):
    print(f"{len(failed)} windows failed; re-run this cell to retry them:", failed.window_id.tolist()[:10])

In [ ]:
# Cell 8B (replacement): Earth Engine auth + resumable batch export to Drive (S5P + S2, all windows)
# Needs Cells 1-7 run first. Safe to re-run any time (after Colab disconnects): finished exports are never restarted.
# State lives in 3 places: Earth Engine's task list, the files in Drive, and CHECKPOINT (json in OUTPUT_DIR).
import json, re, time, threading, concurrent.futures, importlib.util
from collections import Counter
import requests

# ----------------------------- settings (edit here only) -----------------------------
S5P_FOLDER = "Chennai_v3_s5p_composites"   # top-level My Drive folders (Export cannot write into sub-folders)
S2_FOLDER  = "Chennai_v3_s2_composites"
SUBMIT_WORKERS      = 4          # parallel image builds / task starts
MAX_INFLIGHT        = 1000       # max queued+running tasks kept at once (EE queue limit is 3000). >= 876 means
                                 # everything is queued in one go and you can close Colab. Lower it to feed gradually.
POLL_SECONDS        = 60
STALL_MINUTES       = 120        # no progress for this long while tasks are in flight -> stop with an error
MAX_TASK_ATTEMPTS   = 3          # per task (submission or transient task failure)
HARD_FAIL_LIMIT     = 3          # this many non-retryable task failures -> stop immediately
LOST_AFTER_SECONDS  = 600        # checkpoint says submitted but EE has no such task for this long -> resubmit
SUBMIT_TIMEOUT_S    = 1800       # one submission batch must finish within this
EE_DEADLINE_MS      = 120_000    # timeout for each Earth Engine call
WAIT_FOR_COMPLETION = True       # False = return as soon as everything is queued
AUTO_STAMP          = True       # when all exports are complete, run stamp_exports() automatically
RESET_CHECKPOINT    = False      # True = ignore the saved checkpoint (EE task list and Drive files still count)
CHECKPOINT = os.path.join(CONFIG["OUTPUT_DIR"], "export_checkpoint.json")

# ----------------------------- preconditions (fail fast) -----------------------------
_need = ["ee", "EE_PROJECT", "CONFIG", "REGION", "TIME_WINDOWS", "CRS", "S5P_TRANSFORM", "S2_TRANSFORM",
         "S5P_W", "S5P_H", "S2_W", "S2_H", "GASES", "S5P_DESC", "S2_DESC", "NODATA", "S5P_DIR", "S2_DIR",
         "MANIFEST", "BASE_DIR", "get_s5p_composite", "get_s2_composite", "is_v2_file"]
_missing = [n for n in _need if n not in globals()]
if _missing:
    raise RuntimeError(f"Run Cells 1-7 first. Missing from the notebook state: {_missing}")
if not TIME_WINDOWS:
    raise RuntimeError("TIME_WINDOWS is empty (check Cell 3).")
print(f"Region {REGION} | {len(TIME_WINDOWS)} windows -> {2 * len(TIME_WINDOWS)} export tasks")
print(f"Drive folders: {S5P_FOLDER} (S5P), {S2_FOLDER} (S2)")

# ----------------------------- errors and helpers -----------------------------
class ExportAbort(RuntimeError):
    """Raised for anything that must stop the run immediately."""

_AUTH_PAT = re.compile(r"insufficient authentication scopes|ACCESS_TOKEN_SCOPE_INSUFFICIENT|PERMISSION_DENIED|HttpError 403", re.I)
_TRANSIENT_PAT = re.compile(r"429|too many|rate limit|quota|timed? ?out|deadline|unavailable|internal|backend|"
                            r"connection|reset|temporar|try again|50[0234]", re.I)
_TRANSIENT_TASK_PAT = re.compile(r"internal error|try again|timed out|timeout|unavailable|backend|"
                                 r"too many concurrent|rate limit|temporar|aborted|deadline", re.I)
_TERMINAL = ("completed", "no_data", "failed_hard")

abort = threading.Event()
_lock = threading.RLock()

def _is_auth_err(e):
    return bool(_AUTH_PAT.search(str(e)))

def _retry(fn, tries=5):
    for k in range(1, tries + 1):
        try:
            return fn()
        except Exception as e:
            if _is_auth_err(e):
                abort.set()
                raise ExportAbort(f"Permission/scope error from Earth Engine: {str(e)[:200]}") from e
            if "already" in str(e).lower() and "start" in str(e).lower():
                return None                      # task was already started by an earlier attempt
            if k == tries or not _TRANSIENT_PAT.search(str(e)):
                raise
            time.sleep(min(60, 2 ** k))

# ----------------------------- authentication (verified by a real export start) -----------------------------
def _saved_token_scopes():
    try:
        from google.auth.transport.requests import Request
        c = ee.data.get_persistent_credentials()
        if not c.valid:
            c.refresh(Request())
        r = requests.get("https://oauth2.googleapis.com/tokeninfo", params={"access_token": c.token}, timeout=20)
        return sorted(r.json().get("scope", "").split())
    except Exception as e:
        return f"unknown ({type(e).__name__}: {str(e)[:80]})"

def _probe_write():
    # Same call the real exports make (real S5P image, real grid), started and cancelled at once.
    w0 = TIME_WINDOWS[0]
    img, _ = get_s5p_composite(w0["start_date"], w0["end_date"])
    if img is None:
        raise ExportAbort(f"No S5P data in the first window {w0['window_id']}; cannot run the export self-test.")
    t = ee.batch.Export.image.toDrive(
        image=img, description="auth_probe", folder=S5P_FOLDER, fileNamePrefix="auth_probe", crs=CRS,
        crsTransform=S5P_TRANSFORM, dimensions=f"{S5P_W}x{S5P_H}", maxPixels=1e9, fileFormat="GeoTIFF")
    t.start()
    try:
        ee.data.cancelTask(t.id)
    except Exception:
        pass

def _init_current():
    ee.Initialize(project=EE_PROJECT)

def _init_colab():
    from google.colab import auth as colab_auth
    import google.auth
    colab_auth.authenticate_user()
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    ee.Initialize(credentials=creds, project=EE_PROJECT)

def _init_fresh():
    print("Open the link, choose your account and TICK EVERY permission box, then paste the code.")
    ee.Authenticate(scopes=["https://www.googleapis.com/auth/earthengine",
                            "https://www.googleapis.com/auth/devstorage.full_control"],
                    auth_mode="notebook", force=True)
    ee.Initialize(project=EE_PROJECT)

def ensure_export_auth():
    methods = [("existing credentials", _init_current)]
    if importlib.util.find_spec("google.colab") is not None:
        methods.append(("Colab sign-in (authenticate_user)", _init_colab))
    methods.append(("fresh Earth Engine sign-in", _init_fresh))
    for label, init in methods:
        try:
            init()
            try:
                ee.data.setDeadline(EE_DEADLINE_MS)
            except Exception:
                pass
            _probe_write()
            print(f"✓ Export permission verified via: {label}")
            return
        except ExportAbort:
            raise
        except Exception as e:
            if not _is_auth_err(e):
                raise ExportAbort(f"Non-permission error during '{label}': {type(e).__name__}: {str(e)[:300]}") from e
            print(f"✗ '{label}' cannot start exports: {str(e)[:160]}")
    raise ExportAbort(
        "No credential could start an Earth Engine export (ACCESS_TOKEN_SCOPE_INSUFFICIENT / PERMISSION_DENIED).\n"
        f"  Scopes on the saved EE token: {_saved_token_scopes()}\n"
        "  Check: (1) the 'earthengine' box was ticked on the consent screen, (2) you signed in with the account\n"
        f"  that owns project '{EE_PROJECT}', (3) the Earth Engine API is enabled for that project.\n"
        "  If it still fails, use Cell 8A (direct download), which only needs read access.")

# ----------------------------- work items and checkpoint -----------------------------
ITEMS = []
for _w in TIME_WINDOWS:
    for _kind in ("s5p", "s2"):
        ITEMS.append({"kind": _kind, "wid": _w["window_id"], "d0": _w["start_date"], "d1": _w["end_date"],
                      "desc": f"{_kind}_{_w['window_id']}_{REGION}", "prefix": f"{_kind}_{_w['window_id']}",
                      "folder": S5P_FOLDER if _kind == "s5p" else S2_FOLDER})

def _load_ck():
    if not RESET_CHECKPOINT and os.path.exists(CHECKPOINT):
        try:
            with open(CHECKPOINT) as f:
                d = json.load(f)
            if isinstance(d.get("items"), dict):
                return d
        except Exception as e:
            print(f"Checkpoint unreadable ({e}); starting a fresh one (EE tasks and Drive files still count).")
    return {"version": 1, "region": REGION, "items": {}}

ck = _load_ck()

def _save_ck():
    with _lock:
        os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
        tmp = CHECKPOINT + ".tmp"
        with open(tmp, "w") as f:
            json.dump(ck, f)
        os.replace(tmp, CHECKPOINT)

def _set(desc, **kw):
    with _lock:
        rec = ck["items"].setdefault(desc, {"attempts": 0})
        rec.update(kw)
        rec["ts"] = time.time()

def _state(it):
    return ck["items"].get(it["desc"], {}).get("state")

_DRIVE_ROOT = os.path.dirname(BASE_DIR) if str(BASE_DIR).startswith("/content/drive") else None
if _DRIVE_ROOT is None:
    print("Note: Drive is not mounted here, so finished files are tracked only via Earth Engine + the checkpoint.")

def _in_drive(it):
    return bool(_DRIVE_ROOT) and os.path.exists(os.path.join(_DRIVE_ROOT, it["folder"], it["prefix"] + ".tif"))

def _ee_tasks():
    out = {}
    for t in sorted(ee.data.getTaskList(), key=lambda t: t.get("creation_timestamp_ms", 0)):
        if t.get("description"):
            out[t["description"]] = t          # newest task per description wins
    return out

# ----------------------------- submission -----------------------------
def _submit(it):
    if abort.is_set():
        return "skipped"
    desc = it["desc"]
    with _lock:
        _set(desc, attempts=ck["items"].get(desc, {}).get("attempts", 0) + 1)

    def build():
        if it["kind"] == "s5p":
            img, _ = get_s5p_composite(it["d0"], it["d1"])
            return img, S5P_TRANSFORM, S5P_W, S5P_H
        return get_s2_composite(it["d0"], it["d1"]), S2_TRANSFORM, S2_W, S2_H

    img, tr, W, H = _retry(build)
    if img is None:
        _set(desc, state="no_data")
        return "no_data"
    task = ee.batch.Export.image.toDrive(
        image=img, description=desc, folder=it["folder"], fileNamePrefix=it["prefix"], crs=CRS,
        crsTransform=tr, dimensions=f"{W}x{H}", maxPixels=1e9, fileFormat="GeoTIFF")
    _retry(task.start)
    _set(desc, state="submitted", task_id=task.id, error="")
    return "submitted"

def _submit_batch(batch):
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=SUBMIT_WORKERS)
    futs = {ex.submit(_submit, it): it for it in batch}
    done, not_done = concurrent.futures.wait(futs, timeout=SUBMIT_TIMEOUT_S)
    ex.shutdown(wait=False, cancel_futures=True)
    errs = []
    for f in done:
        it = futs[f]
        try:
            f.result()
        except ExportAbort:
            raise
        except Exception as e:
            msg = f"{type(e).__name__}: {str(e)[:150]}"
            errs.append((it["desc"], msg))
            _set(it["desc"], error=msg)
    _save_ck()
    if not_done:
        abort.set()
        raise ExportAbort(f"{len(not_done)} submissions did not finish within {SUBMIT_TIMEOUT_S}s (Earth Engine hanging?). "
                          "Re-run the cell to resume.")
    if errs:
        print(f"  {len(errs)} submissions failed (will retry up to {MAX_TASK_ATTEMPTS}x):")
        for d, m in errs[:3]:
            print("   ", d, m)
        if len(errs) >= 5 and len(errs) >= 0.5 * len(done):
            raise ExportAbort("Most submissions in this batch failed; stopping instead of retrying blindly. "
                              f"First error: {errs[0][0]}: {errs[0][1]}")

# ----------------------------- state refresh -----------------------------
def _refresh():
    tasks = _retry(_ee_tasks)
    now = time.time()
    for it in ITEMS:
        d = it["desc"]
        rec = ck["items"].get(d, {})
        st = rec.get("state")
        if st in _TERMINAL:
            continue
        if _in_drive(it):
            _set(d, state="completed", via="drive")
            continue
        t = tasks.get(d)
        if t is None:
            if st == "submitted" and now - rec.get("ts", 0) > LOST_AFTER_SECONDS:
                _set(d, state="retry", error="task not found at Earth Engine")
        else:
            created = t.get("creation_timestamp_ms", 0) / 1000.0
            stale_old_task = st == "submitted" and created < rec.get("ts", 0) - 30
            if not stale_old_task:
                s = t.get("state")
                if s == "COMPLETED":
                    _set(d, state="completed", via="ee")
                elif s in ("READY", "RUNNING", "CANCEL_REQUESTED", "UNSUBMITTED"):
                    if st != "submitted":
                        _set(d, state="submitted", task_id=t.get("id"))
                elif s in ("FAILED", "CANCELLED"):
                    msg = (t.get("error_message") or "").strip()
                    if _is_auth_err(msg):
                        abort.set()
                        raise ExportAbort(f"{d} failed with a permission error: {msg[:200]}")
                    if s == "FAILED" and msg and not _TRANSIENT_TASK_PAT.search(msg):
                        _set(d, state="failed_hard", error=msg[:300])
                    else:
                        _set(d, state="retry", error=msg[:300] or s)
        rec = ck["items"].get(d, {})
        if rec.get("state") == "retry" and rec.get("attempts", 0) >= MAX_TASK_ATTEMPTS:
            _set(d, state="failed_hard", error=f"retries exhausted ({MAX_TASK_ATTEMPTS}): {rec.get('error', '')}"[:300])
    _save_ck()

def _snapshot():
    inflight = [it for it in ITEMS if _state(it) == "submitted"]
    pending = [it for it in ITEMS if _state(it) not in _TERMINAL + ("submitted",)]
    return inflight, pending

def _hard_failures():
    return [(it["desc"], ck["items"][it["desc"]].get("error", "")) for it in ITEMS if _state(it) == "failed_hard"]

# ----------------------------- driver -----------------------------
def run_export():
    abort.clear()
    ensure_export_auth()
    existing = Counter(_state(it) for it in ITEMS)
    print(f"Checkpoint: {dict(existing)}")
    t0 = time.time()
    last_key, last_change = None, time.time()
    try:
        while True:
            _refresh()
            hard = _hard_failures()
            if len(hard) >= HARD_FAIL_LIMIT:
                raise ExportAbort(f"{len(hard)} tasks failed with non-retryable errors, stopping. First errors:\n" +
                                  "\n".join(f"  {d}: {m}" for d, m in hard[:5]))
            inflight, pending = _snapshot()
            room = MAX_INFLIGHT - len(inflight)
            if pending and room > 0:
                print(f"Submitting {min(room, len(pending))} of {len(pending)} pending tasks ...")
                _submit_batch(pending[:room])
                inflight, pending = _snapshot()
            c = Counter(_state(it) for it in ITEMS)
            print(f"[{(time.time() - t0) / 60:5.1f} min] done {c['completed']} | queued/running {len(inflight)} | "
                  f"to submit {len(pending)} | no data {c['no_data']} | failed {c['failed_hard']}")
            if not inflight and not pending:
                break
            if not pending and not WAIT_FOR_COMPLETION:
                print("Everything is queued. Closing Colab is safe; re-run this cell later to check/resume.")
                return
            key = (c["completed"], c["failed_hard"], len(inflight), len(pending),
                   sum(r.get("attempts", 0) for r in ck["items"].values()))
            if key != last_key:
                last_key, last_change = key, time.time()
            elif inflight and time.time() - last_change > STALL_MINUTES * 60:
                raise ExportAbort(f"No progress for {STALL_MINUTES} min with {len(inflight)} tasks in flight. "
                                  "Check https://code.earthengine.google.com/tasks, then re-run this cell.")
            time.sleep(POLL_SECONDS)
    except KeyboardInterrupt:
        _save_ck()
        print("Interrupted. Tasks already submitted keep running on Google's side; re-run this cell to resume.")
        raise
    except ExportAbort:
        _save_ck()
        raise
    _save_ck()
    c = Counter(_state(it) for it in ITEMS)
    hard = _hard_failures()
    print(f"FINISHED: completed {c['completed']} | no data {c['no_data']} | failed {c['failed_hard']} of {len(ITEMS)}")
    if hard:
        raise ExportAbort(f"{len(hard)} tasks failed permanently:\n" + "\n".join(f"  {d}: {m}" for d, m in hard[:10]))
    if AUTO_STAMP:
        stamp_exports()

# ----------------------------- stamp: copy into data_v3_<region>/, tag as v2, write manifest for Cell 9 -----------------------------
def stamp_exports():
    drive_root = os.path.dirname(BASE_DIR)
    rows, missing = [], []
    for w in tqdm(TIME_WINDOWS, desc="Stamping"):
        wid = w["window_id"]
        rec = {**w, "s5p_file": f"s5p_{wid}.tif", "s2_file": f"s2_{wid}.tif", "pipeline": "v2"}
        for kind, folder, out_dir, desc in (("s5p", S5P_FOLDER, S5P_DIR, S5P_DESC), ("s2", S2_FOLDER, S2_DIR, S2_DESC)):
            src_p = os.path.join(drive_root, folder, f"{kind}_{wid}.tif")
            dst_p = os.path.join(out_dir, f"{kind}_{wid}.tif")
            a = None
            if is_v2_file(dst_p):
                with rasterio.open(dst_p) as s:
                    a = s.read()
            elif os.path.exists(src_p):
                with rasterio.open(src_p) as s:
                    a, prof = s.read().astype("float32"), s.profile
                assert a.shape[0] == len(desc), f"{src_p}: {a.shape[0]} bands, expected {len(desc)}"
                a[~np.isfinite(a)] = NODATA
                prof.update(dtype="float32", nodata=NODATA, compress="deflate", driver="GTiff")
                with rasterio.open(dst_p, "w", **prof) as d:
                    d.write(a)
                    d.descriptions = tuple(desc)
                    d.update_tags(pipeline="v2", created=datetime.datetime.now(datetime.timezone.utc).isoformat(), region=REGION)
            if a is not None:
                rec[f"{kind}_status"] = "ok"
            elif ck["items"].get(f"{kind}_{wid}_{REGION}", {}).get("state") == "no_data":
                rec[f"{kind}_status"] = "no_s5p_data" if kind == "s5p" else "cloud_gap"
            else:
                rec[f"{kind}_status"] = "missing"
                missing.append(f"{kind}_{wid}")
            if kind == "s5p" and a is not None:
                for i, g in enumerate(GASES):
                    rec[f"valid_{g}"] = float(np.mean(a[i] != NODATA))
            if kind == "s2" and a is not None:
                rec["s2_clear"] = float(np.mean(a[0] != NODATA))
        ok5, ok2 = rec["s5p_status"] == "ok", rec["s2_status"] == "ok"
        rec["status"] = "complete" if ok5 and ok2 else ("s5p_only" if ok5 else "failed")
        rows.append(rec)
    pd.DataFrame(rows).to_csv(MANIFEST, index=False)
    print("manifest written:", MANIFEST, "|", pd.Series([r["status"] for r in rows]).value_counts().to_dict())
    if missing:
        print(f"WARNING: {len(missing)} expected files not in Drive yet (sync lag or failed export), e.g. {missing[:5]}. "
              "Wait a minute and call stamp_exports() again.")

run_export()

Region chennai | 438 windows -> 876 export tasks
Drive folders: Chennai_v3_s5p_composites (S5P), Chennai_v3_s2_composites (S2)


✗ 'existing credentials' cannot start exports: Request had insufficient authentication scopes.


✓ Export permission verified via: Colab sign-in (authenticate_user)
Checkpoint: {None: 876}
Submitting 876 of 876 pending tasks ...
[  2.4 min] done 0 | queued/running 776 | to submit 0 | no data 100 | failed 0
[  3.5 min] done 25 | queued/running 751 | to submit 0 | no data 100 | failed 0
[  4.5 min] done 30 | queued/running 746 | to submit 0 | no data 100 | failed 0
[  5.6 min] done 41 | queued/running 735 | to submit 0 | no data 100 | failed 0
[  6.6 min] done 47 | queued/running 729 | to submit 0 | no data 100 | failed 0
[  7.7 min] done 56 | queued/running 720 | to submit 0 | no data 100 | failed 0
[  8.7 min] done 66 | queued/running 710 | to submit 0 | no data 100 | failed 0
[  9.7 min] done 75 | queued/running 701 | to submit 0 | no data 100 | failed 0
[ 10.8 min] done 80 | queued/running 696 | to submit 0 | no data 100 | failed 0
[ 11.8 min] done 90 | queued/running 686 | to submit 0 | no data 100 | failed 0
[ 12.8 min] done 107 | queued/running 669 | to submit 0 | no data 100

[ 49.0 min] done 383 | queued/running 393 | to submit 0 | no data 100 | failed 0
[ 50.0 min] done 392 | queued/running 384 | to submit 0 | no data 100 | failed 0
[ 51.0 min] done 403 | queued/running 373 | to submit 0 | no data 100 | failed 0
[ 52.0 min] done 410 | queued/running 366 | to submit 0 | no data 100 | failed 0
[ 53.1 min] done 417 | queued/running 359 | to submit 0 | no data 100 | failed 0
[ 54.1 min] done 427 | queued/running 349 | to submit 0 | no data 100 | failed 0
[ 55.1 min] done 434 | queued/running 342 | to submit 0 | no data 100 | failed 0
[ 56.1 min] done 442 | queued/running 334 | to submit 0 | no data 100 | failed 0
[ 57.1 min] done 450 | queued/running 326 | to submit 0 | no data 100 | failed 0
[ 58.2 min] done 459 | queued/running 317 | to submit 0 | no data 100 | failed 0
[ 59.2 min] done 467 | queued/running 309 | to submit 0 | no data 100 | failed 0
[ 60.2 min] done 476 | queued/running 300 | to submit 0 | no data 100 | failed 0
[ 61.2 min] done 484 | queue

[106.0 min] done 730 | queued/running 46 | to submit 0 | no data 100 | failed 0
[107.0 min] done 736 | queued/running 40 | to submit 0 | no data 100 | failed 0
[108.1 min] done 740 | queued/running 36 | to submit 0 | no data 100 | failed 0
[109.1 min] done 747 | queued/running 29 | to submit 0 | no data 100 | failed 0
[110.1 min] done 753 | queued/running 23 | to submit 0 | no data 100 | failed 0
[111.1 min] done 757 | queued/running 19 | to submit 0 | no data 100 | failed 0
[112.1 min] done 764 | queued/running 12 | to submit 0 | no data 100 | failed 0
[113.1 min] done 776 | queued/running 0 | to submit 0 | no data 100 | failed 0
FINISHED: completed 776 | no data 100 | failed 0 of 876


Stamping:   0%|          | 0/438 [00:00<?, ?it/s]

manifest written: /content/drive/MyDrive/Chennai_Satellite_Dataset/data_v3_chennai/dataset_manifest.csv | {'failed': 438}


In [ ]:
# Cell 8C: Monitor Cloud Cluster Task Status
# Run this anytime to check how many tasks are Completed / Running on Google's cloud servers
import pandas as pd

tasks = ee.data.getTaskList()
df_tasks = pd.DataFrame(tasks)
if not df_tasks.empty and "state" in df_tasks.columns:
    print("Current GEE Cloud Cluster Status:")
    print(df_tasks["state"].value_counts().to_string())
    print("\nRecent Tasks:")
    cols = [c for c in ["description", "state", "creation_timestamp_ms", "update_timestamp_ms"] if c in df_tasks.columns]
    print(df_tasks[cols].head(10).to_string())
else:
    print("No active or recent tasks found in GEE queue.")


Current GEE Cloud Cluster Status:
state
COMPLETED    776
CANCELLED      1

Recent Tasks:
            description      state  creation_timestamp_ms  update_timestamp_ms
0  s5p_20241225_chennai  COMPLETED          1792307690867        1792314322838
1   s2_20241215_chennai  COMPLETED          1792307690469        1792314318955
2  s5p_20241220_chennai  COMPLETED          1792307690463        1792314316522
3  s5p_20241215_chennai  COMPLETED          1792307690378        1792314310984
4  s5p_20241130_chennai  COMPLETED          1792307689912        1792314299722
5  s5p_20241210_chennai  COMPLETED          1792307689900        1792314308761
6  s5p_20241205_chennai  COMPLETED          1792307689779        1792314286801
7  s5p_20241125_chennai  COMPLETED          1792307689380        1792314301628
8   s2_20241115_chennai  COMPLETED          1792307689132        1792314284387
9   s2_20241120_chennai  COMPLETED          1792307689121        1792314305558


In [ ]:

from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


### 🔍 Dataset Verification and Health Check
This cell verifies the completeness and internal integrity of the S5P and S2 composite imagery stored under your `Satelite Dataset-UPDATED` Google Drive folder.

In [ ]:
import os

def find_folder_and_verify(search_name="Satelite Dataset-UPDATED"):
    print(f"Searching for '{search_name}' folder in your mounted Google Drive...")
    drive_path = "/content/drive/MyDrive"
    matched_paths = []

    # Walk through directories in My Drive to find the exact name match
    for root, dirs, files in os.walk(drive_path):
        if search_name in dirs:
            full_path = os.path.join(root, search_name)
            matched_paths.append(full_path)
            print(f"🎯 Found folder at: {full_path}")

    if not matched_paths:
        print(f"❌ Could not find any folder named '{search_name}' under /content/drive/MyDrive.")
        print("Please check if the folder name is spelled exactly as specified or if it is in Shared Drives.")
        return

    # Check contents of the found directory
    target_folder = matched_paths[0]
    print(f"\nProceeding with verification on: {target_folder}")

    subfolders = os.listdir(target_folder)
    print(f"Contents of '{search_name}': {subfolders}")

    for sub in subfolders:
        sub_path = os.path.join(target_folder, sub)
        if os.path.isdir(sub_path):
            tifs = [f for f in os.listdir(sub_path) if f.endswith('.tif')]
            print(f"📁 Subfolder '{sub}': Found {len(tifs)} GeoTIFF (.tif) files.")

find_folder_and_verify()

Searching for 'Satelite Dataset-UPDATED' folder in your mounted Google Drive...
🎯 Found folder at: /content/drive/MyDrive/Satelite Dataset-UPDATED

Proceeding with verification on: /content/drive/MyDrive/Satelite Dataset-UPDATED
Contents of 'Satelite Dataset-UPDATED': ['Chennai_v3_s2_composites', 'Chennai_v3_s5p_composites (1)']
📁 Subfolder 'Chennai_v3_s2_composites': Found 337 GeoTIFF (.tif) files.
📁 Subfolder 'Chennai_v3_s5p_composites (1)': Found 437 GeoTIFF (.tif) files.


In [ ]:
import os
import glob
import rasterio
import pandas as pd
import numpy as np

# Corrected paths based on discovery results
UPDATED_DRIVE_ROOT = "/content/drive/MyDrive/Satelite Dataset-UPDATED"
s5p_dir_corrected = os.path.join(UPDATED_DRIVE_ROOT, "Chennai_v3_s5p_composites (1)")
s2_dir_corrected = os.path.join(UPDATED_DRIVE_ROOT, "Chennai_v3_s2_composites")

def verify_geotiffs(directory, expected_bands, dataset_name):
    if not os.path.exists(directory):
        print(f"❌ Directory not found: {directory}")
        return

    tiffs = sorted(glob.glob(os.path.join(directory, "*.tif")))
    if not tiffs:
        print(f"\n⚠️ No GeoTIFFs found in {dataset_name} folder.")
        return

    print(f"\n=== 📋 Validating {dataset_name} GeoTIFFs ({len(tiffs)} files) ===")
    corrupted = []
    mismatched_bands = []
    stats = []

    for filepath in tiffs[:15]: # Profile the first 15 for deep structural analysis
        filename = os.path.basename(filepath)
        try:
            with rasterio.open(filepath) as src:
                bands = src.count
                meta = src.profile
                tags = src.tags()

                if bands != expected_bands:
                    mismatched_bands.append((filename, bands))
                else:
                    stats.append({
                        "file": filename,
                        "bands": bands,
                        "width": src.width,
                        "height": src.height,
                        "nodata": src.nodata,
                        "pipeline_tag": tags.get("pipeline", "none"),
                        "crs": str(src.crs)
                    })
        except Exception as e:
            corrupted.append((filename, str(e)))

    # Overall reports
    if corrupted:
        print(f"❌ Corrupted/Unreadable Files ({len(corrupted)}):", corrupted)
    else:
        print("✅ No corrupted files detected in sampled subset.")

    if mismatched_bands:
        print(f"⚠️ Band mismatch found in ({len(mismatched_bands)}) files! Expected {expected_bands}:", mismatched_bands)
    else:
        print(f"✅ All sampled files contain exactly {expected_bands} bands as expected.")

    if stats:

        df_stats = pd.DataFrame(stats)
        print(f"\nSample Profiles for {dataset_name}:")
        display(df_stats.head(5))

# Run the verification on actual folders
verify_geotiffs(s5p_dir_corrected, expected_bands=6, dataset_name="Sentinel-5P")
verify_geotiffs(s2_dir_corrected, expected_bands=12, dataset_name="Sentinel-2")


=== 📋 Validating Sentinel-5P GeoTIFFs (437 files) ===
✅ No corrupted files detected in sampled subset.
✅ All sampled files contain exactly 6 bands as expected.

Sample Profiles for Sentinel-5P:


,file,bands,width,height,nodata,pipeline_tag,crs
0,s5p_20190106.tif,6,10,10,None,none,EPSG:4326
1,s5p_20190111.tif,6,10,10,None,none,EPSG:4326
2,s5p_20190116.tif,6,10,10,None,none,EPSG:4326
3,s5p_20190121.tif,6,10,10,None,none,EPSG:4326
4,s5p_20190126.tif,6,10,10,None,none,EPSG:4326



=== 📋 Validating Sentinel-2 GeoTIFFs (337 files) ===
✅ No corrupted files detected in sampled subset.
✅ All sampled files contain exactly 12 bands as expected.

Sample Profiles for Sentinel-2:


,file,bands,width,height,nodata,pipeline_tag,crs
0,s2_20190101.tif,12,80,80,None,none,EPSG:4326
1,s2_20190111.tif,12,80,80,None,none,EPSG:4326
2,s2_20190126.tif,12,80,80,None,none,EPSG:4326
3,s2_20190131.tif,12,80,80,None,none,EPSG:4326
4,s2_20190205.tif,12,80,80,None,none,EPSG:4326


In [ ]:
import os, glob, json, datetime
import numpy as np, pandas as pd, rasterio
from tqdm.auto import tqdm

DRIVE_EXPORT_ROOT = "/content/drive/MyDrive/Satelite Dataset-UPDATED"
MYDRIVE = "/content/drive/MyDrive"

def find_dirs(prefix):
    cands = glob.glob(os.path.join(DRIVE_EXPORT_ROOT, prefix + "*")) + glob.glob(os.path.join(MYDRIVE, prefix + "*"))
    return [d for d in dict.fromkeys(cands) if os.path.isdir(d)]

SOURCES = {"s5p": find_dirs("Chennai_v3_s5p_composites"), "s2": find_dirs("Chennai_v3_s2_composites")}
for k, dirs in SOURCES.items():
    for d in dirs:
        n = len([f for f in os.listdir(d) if f.lower().endswith((".tif", ".tiff"))])
        print(f"{k}: {d} | {n} tiffs")
    assert dirs, f"no {k} source folder visible"

# checkpoint: reuse the in-memory one, otherwise reload from disk
if "ck" not in globals() or not isinstance(ck, dict):
    ck = json.load(open(CHECKPOINT)) if os.path.exists(CHECKPOINT) else {"items": {}}

def find_src(kind, wid):
    for d in SOURCES[kind]:
        p = os.path.join(d, f"{kind}_{wid}.tif")
        if os.path.exists(p):
            return p
    return None

def stamp_exports():
    rows, missing = [], []
    for w in tqdm(TIME_WINDOWS, desc="Stamping"):
        wid = w["window_id"]
        rec = {**w, "s5p_file": f"s5p_{wid}.tif", "s2_file": f"s2_{wid}.tif", "pipeline": "v2"}
        for kind, out_dir, desc in (("s5p", S5P_DIR, S5P_DESC), ("s2", S2_DIR, S2_DESC)):
            src_p = find_src(kind, wid)
            dst_p = os.path.join(out_dir, f"{kind}_{wid}.tif")
            a = None
            if is_v2_file(dst_p):
                with rasterio.open(dst_p) as s:
                    a = s.read()
            elif src_p:
                with rasterio.open(src_p) as s:
                    a, prof = s.read().astype("float32"), s.profile
                assert a.shape[0] == len(desc), f"{src_p}: {a.shape[0]} bands, expected {len(desc)}"
                a[~np.isfinite(a)] = NODATA
                prof.update(dtype="float32", nodata=NODATA, compress="deflate", driver="GTiff")
                os.makedirs(out_dir, exist_ok=True)
                with rasterio.open(dst_p, "w", **prof) as d:
                    d.write(a)
                    d.descriptions = tuple(desc)
                    d.update_tags(pipeline="v2", region=REGION,
                                  created=datetime.datetime.now(datetime.timezone.utc).isoformat())
            if a is not None:
                rec[f"{kind}_status"] = "ok"
            elif ck["items"].get(f"{kind}_{wid}_{REGION}", {}).get("state") == "no_data":
                rec[f"{kind}_status"] = "no_s5p_data" if kind == "s5p" else "cloud_gap"
            else:
                rec[f"{kind}_status"] = "missing"
                missing.append(f"{kind}_{wid}")
            if kind == "s5p" and a is not None:
                for i, g in enumerate(GASES):
                    rec[f"valid_{g}"] = float(np.mean(a[i] != NODATA))
            if kind == "s2" and a is not None:
                rec["s2_clear"] = float(np.mean(a[0] != NODATA))
        ok5, ok2 = rec["s5p_status"] == "ok", rec["s2_status"] == "ok"
        rec["status"] = "complete" if ok5 and ok2 else ("s5p_only" if ok5 else "failed")
        rows.append(rec)
    pd.DataFrame(rows).to_csv(MANIFEST, index=False)
    print("Manifest written:", MANIFEST)
    print(pd.Series([r["status"] for r in rows]).value_counts())
    print("Still missing:", len(missing), missing[:10])
    return missing

missing = stamp_exports()
assert not missing, "Run the two-file recovery cell, then re-run this cell."
print("DATASET COMPLETE")

s5p: /content/drive/MyDrive/Satelite Dataset-UPDATED/Chennai_v3_s5p_composites (1) | 437 tiffs
s2: /content/drive/MyDrive/Satelite Dataset-UPDATED/Chennai_v3_s2_composites | 337 tiffs


Stamping:   0%|          | 0/438 [00:00<?, ?it/s]

Manifest written: /content/drive/MyDrive/Chennai_Satellite_Dataset/data_v3_chennai/dataset_manifest.csv
complete    338
s5p_only    100
Name: count, dtype: int64
Still missing: 0 []
DATASET COMPLETE


In [ ]:
# Cell 10: recover the 2 files missing from Drive (s5p_20190101, s2_20190106) and fix the manifest.
# Needs Cells 1-6 in this runtime (EE initialised, Drive mounted). Safe to re-run.
import glob
MISSING = [("s5p", "20190101"), ("s2", "20190106")]
SPEC = {"s5p": (S5P_DIR, S5P_DESC, S5P_TRANSFORM, S5P_W, S5P_H),
        "s2":  (S2_DIR,  S2_DESC,  S2_TRANSFORM,  S2_W,  S2_H)}
WIN = {w["window_id"]: w for w in TIME_WINDOWS}

def stamp_copy(src_p, dst_p, desc):
    # Same steps as stamp_exports(): float32, -9999 nodata, band names, pipeline=v2 tag.
    with rasterio.open(src_p) as s:
        a, prof = s.read().astype("float32"), s.profile
    assert a.shape[0] == len(desc), f"{src_p}: {a.shape[0]} bands, expected {len(desc)}"
    a[~np.isfinite(a)] = NODATA
    prof.update(dtype="float32", nodata=NODATA, compress="deflate", driver="GTiff")
    tmp = dst_p + ".tmp"
    with rasterio.open(tmp, "w", **prof) as d:
        d.write(a)
        d.descriptions = tuple(desc)
        d.update_tags(pipeline="v2", created=datetime.datetime.now(datetime.timezone.utc).isoformat(), region=REGION)
    os.replace(tmp, dst_p)
    return a

got = {}
for kind, wid in MISSING:
    out_dir, desc, tr, W, H = SPEC[kind]
    name = f"{kind}_{wid}.tif"
    dst = os.path.join(out_dir, name)
    if is_v2_file(dst):
        with rasterio.open(dst) as s:
            a, how = s.read(), "already in place"
    else:
        # 1) The EE task COMPLETED, so the file is most likely in a duplicate Drive folder Export created.
        hits = [p for p in glob.glob(f"/content/drive/MyDrive/Chennai_v3_*/{name}")
                + glob.glob(f"/content/drive/MyDrive/*/Chennai_v3_*/{name}") if os.path.abspath(p) != dst]
        if hits:
            a, how = stamp_copy(hits[0], dst, desc), f"copied from {hits[0]}"
        else:
            # 2) Otherwise build the same composite again and download it directly (Cell 8A path).
            w = WIN[wid]
            img = (get_s5p_composite(w["start_date"], w["end_date"])[0] if kind == "s5p"
                   else get_s2_composite(w["start_date"], w["end_date"]))
            assert img is not None, f"{name}: Earth Engine has no data for this window"
            a, how = download(img, dst, tr, W, H, desc), "downloaded directly from Earth Engine"
    assert a.shape == (len(desc), H, W), f"{name}: shape {a.shape}"
    assert is_v2_file(dst), f"{name}: not tagged v2 / nodata -9999"
    if kind == "s5p":
        v = a[:3]
        print(f"{name}: {how} | valid " + ", ".join(f"{g} {np.mean(v[i] != NODATA):.0%}" for i, g in enumerate(GASES))
              + f" | exact zeros {np.mean(v[v != NODATA] == 0):.2%}")
    else:
        print(f"{name}: {how} | clear {np.mean(a[0] != NODATA):.0%}")
    got[(kind, wid)] = a

# Update only these two rows of the manifest (same columns stamp_exports() writes).
m = pd.read_csv(MANIFEST, dtype={"window_id": str})
for (kind, wid), a in got.items():
    i = m.index[m.window_id == wid][0]
    m.loc[i, f"{kind}_status"] = "ok"
    if kind == "s5p":
        for j, g in enumerate(GASES):
            m.loc[i, f"valid_{g}"] = float(np.mean(a[j] != NODATA))
    else:
        m.loc[i, "s2_clear"] = float(np.mean(a[0] != NODATA))
    ok5, ok2 = m.loc[i, "s5p_status"] == "ok", m.loc[i, "s2_status"] == "ok"
    m.loc[i, "status"] = "complete" if ok5 and ok2 else ("s5p_only" if ok5 else "failed")
m.to_csv(MANIFEST, index=False)

# Final completeness check: expect 438 S5P + 338 S2 v2 files, manifest complete 338 / s5p_only 100 / failed 0.
n5 = sum(is_v2_file(os.path.join(S5P_DIR, f)) for f in m.s5p_file[m.s5p_status == "ok"])
n2 = sum(is_v2_file(os.path.join(S2_DIR, f)) for f in m.s2_file[m.s2_status == "ok"])
print(m.status.value_counts().to_dict(), "| S5P v2 files", n5, "/ 438 | S2 v2 files", n2, "/ 338")
assert len(m) == 438 and n5 == 438 and (m.status == "failed").sum() == 0, "dataset still incomplete"
print("DATASET COMPLETE")


/tmp/ipykernel_524/2769490694.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  dst.update_tags(pipeline="v2", created=datetime.datetime.utcnow().isoformat(), region=REGION)


s5p_20190101.tif: downloaded directly from Earth Engine | valid NO2 100%, CO 99%, SO2 100% | exact zeros 0.00%


s2_20190106.tif: downloaded directly from Earth Engine | clear 62%
{'complete': 338, 's5p_only': 100} | S5P v2 files 438 / 438 | S2 v2 files 338 / 338
DATASET COMPLETE
